In [ ]:
import cv2
import numpy as np
import os

# =============================
# CONFIG
# =============================
INPUT_IMAGE = "20260217_230429.jpg"
OUTPUT_DIR = "/home/chitipat/Documents/Year_3/PracticalDS/crop/images"
PADDING = 20  # pixels padding around each bean

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =============================
# LOAD IMAGE
# =============================
img = cv2.imread(INPUT_IMAGE)
if img is None:
    raise FileNotFoundError(f"Image not found: {INPUT_IMAGE}")

original = img.copy()
h, w = img.shape[:2]
print(f"Image size: {w} x {h}")

# =============================
# PREPROCESSING
# =============================
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Gaussian blur to reduce noise
blurred = cv2.GaussianBlur(gray, (7, 7), 0)

# Threshold - coffee beans are darker than white background
_, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

# Morphological ops for cleanup
kernel = np.ones((5, 5), np.uint8)
thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=2)
thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=1)

# Debug: save threshold image
cv2.imwrite(os.path.join(OUTPUT_DIR, "_debug_thresh.jpg"), thresh)

# =============================
# FIND CONTOURS
# =============================
contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

print(f"Total contours found: {len(contours)}")

# Filter by area - keep only coffee bean sized contours
img_area = h * w
min_area = img_area * 0.0005   # min 0.05% of image
max_area = img_area * 0.02     # max 2% of image

print(f"Area filter: {min_area:.0f} - {max_area:.0f} px²")

valid_contours = []
for cnt in contours:
    area = cv2.contourArea(cnt)
    if min_area < area < max_area:
        valid_contours.append(cnt)

print(f"Valid contours after filter: {len(valid_contours)}")

# =============================
# SORT: top-left to bottom-right (row order)
# =============================
def sort_contours_grid(contours, row_height_factor=0.08):
    """Sort contours in reading order (top-left to bottom-right)"""
    bboxes = [cv2.boundingRect(c) for c in contours]
    # Combine contour + bbox
    combined = sorted(zip(bboxes, contours), key=lambda b: b[0][1])  # sort by y first
    
    # Group into rows
    rows = []
    current_row = [combined[0]]
    row_threshold = img.shape[0] * row_height_factor
    
    for item in combined[1:]:
        if abs(item[0][1] - current_row[0][0][1]) < row_threshold:
            current_row.append(item)
        else:
            rows.append(sorted(current_row, key=lambda b: b[0][0]))  # sort by x within row
            current_row = [item]
    rows.append(sorted(current_row, key=lambda b: b[0][0]))
    
    sorted_contours = []
    for row in rows:
        for bbox, cnt in row:
            sorted_contours.append(cnt)
    return sorted_contours

sorted_conts = sort_contours_grid(valid_contours)

# =============================
# CROP & SAVE
# =============================
debug_img = original.copy()
saved_count = 0

for i, cnt in enumerate(sorted_conts):
    x, y, bw, bh = cv2.boundingRect(cnt)
    
    # Add padding (clamp to image boundary)
    x1 = max(0, x - PADDING)
    y1 = max(0, y - PADDING)
    x2 = min(w, x + bw + PADDING)
    y2 = min(h, y + bh + PADDING)
    
    # Crop
    cropped = original[y1:y2, x1:x2]
    
    # Naming: 4_55.jpg, 4_56.jpg, ...
    filename = f"4_{54+i+1}.jpg"
    filepath = os.path.join(OUTPUT_DIR, filename)
    cv2.imwrite(filepath, cropped, [cv2.IMWRITE_JPEG_QUALITY, 95])
    
    saved_count += 1
    print(f"  [{i+1:02d}] {filename} → ({x1},{y1}) size {x2-x1}x{y2-y1}")
    
    # Draw on debug image
    cv2.rectangle(debug_img, (x1, y1), (x2, y2), (0, 255, 0), 3)
    cv2.putText(debug_img, str(i+1), (x1, y1 - 8),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 2)

# Save annotated debug image
cv2.imwrite(os.path.join(OUTPUT_DIR, "_debug_annotated.jpg"), debug_img)

print(f"\nDone! Saved {saved_count} beans to {OUTPUT_DIR}")

ขนาดรูป: 4000 x 3000
พบ contours ทั้งหมด: 26
Area filter: 6000 - 240000 px²
Contours ที่ผ่านการกรอง: 25
  [01] 4_55.jpg → (223,173) size 155x185
  [02] 4_56.jpg → (1102,200) size 183x158
  [03] 4_57.jpg → (1939,128) size 154x156
  [04] 4_58.jpg → (2756,130) size 150x163
  [05] 4_59.jpg → (3460,173) size 174x163
  [06] 4_60.jpg → (172,623) size 156x182
  [07] 4_61.jpg → (1132,590) size 168x151
  [08] 4_62.jpg → (1971,623) size 179x168
  [09] 4_63.jpg → (2721,608) size 183x154
  [10] 4_64.jpg → (3464,622) size 184x146
  [11] 4_65.jpg → (149,1161) size 193x158
  [12] 4_66.jpg → (1101,1017) size 173x163
  [13] 4_67.jpg → (1964,1088) size 185x138
  [14] 4_68.jpg → (2747,1085) size 181x152
  [15] 4_69.jpg → (3574,1054) size 194x179
  [16] 4_70.jpg → (111,1705) size 183x152
  [17] 4_71.jpg → (1133,1627) size 138x193
  [18] 4_72.jpg → (1942,1572) size 182x163
  [19] 4_73.jpg → (2734,1577) size 214x168
  [20] 4_74.jpg → (3617,1545) size 202x144
  [21] 4_75.jpg → (48,2370) size 170x172
  [22] 4_